# NL2SQL Experiments: Natural Language to SQL Query Generation

**Purpose:** This notebook evaluates Natural Language to SQL (NL2SQL) query generation using different experimental configurations. A Knowledge Graph is built from the PostgreSQL schema, and SQL queries are generated and executed against the database.

**Experiments:**
1. **E1 - Baseline One-Shot:** KG schema without semantics, single-shot SQL generation
2. **E2 - Semantic KG + Multi-Stage LLM:** Schema linking with descriptions, synonyms, multi-stage pipeline
3. **E3 - Execution-Guided Retry:** Same as E2 with one retry on execution failure
4. **E4 - SQL-Specialized Model:** defog/sqlcoder-7b-2, same pipeline as E2/E3 with retry enabled
5. **E5 - Example-Augmented KG:** Few-shot retrieval (20 examples), schema overlap + structural similarity, retry with example context

**Test Data:** Same CSV used for all experiments (`test_data_set/final_test_data.csv`)

**Results:** Saved per experiment to `test_data_set/nl2sql_experiment_{N}_results.csv`

## 1. Setup and Dependencies

Install required packages: PostgreSQL client, Neo4j driver, Ollama client, SQL parser, and utilities.

In [1]:
!pip -q install psycopg2-binary pandas neo4j ollama sqlparse tqdm

## 2. Imports and Configuration

Import libraries and set environment variables for PostgreSQL, Neo4j, and Ollama connections.

In [2]:
import os, json, re, time, hashlib
import threading
import pandas as pd
import psycopg2
import sqlparse
import ollama
from neo4j import GraphDatabase
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import datetime
import decimal
import uuid

# Paths (notebook in jupiter_note_books/, test_data_set is sibling)
_cwd = os.getcwd()
BASE_DIR = os.path.dirname(_cwd) if 'jupiter_note_books' in _cwd else _cwd
CSV_PATH = os.path.join(BASE_DIR, "test_data_set", "final_test_data.csv")
RESULTS_DIR = os.path.join(BASE_DIR, "test_data_set")

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "customer_orders_and_reviews_db")
PG_USER = os.getenv("PG_USER", "postgres")
PG_PASS = os.getenv("PG_PASS", "postgres")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = os.getenv("NEO4J_PASS", "neo4jpassword")

# Ollama
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma3:4b")
OLLAMA_MODEL_E4 = os.getenv("OLLAMA_MODEL_E4", "sqlcoder:7b")  # SQL-specialized; try 'sqlcoder' or 'defog/sqlcoder-7b-2' via Modelfile
TEMPERATURE = 0.1
FETCH_ROWS = 20

# Speed: set to 5 or 10 for quick iteration; None = full 75. Edit or set env MAX_TEST_CASES.
MAX_TEST_CASES = None  # e.g. 5 or 10 for fast runs
if os.getenv("MAX_TEST_CASES"):
    try: MAX_TEST_CASES = int(os.getenv("MAX_TEST_CASES")) or None
    except: pass

# Parallel workers for all 75 tests. 2-4 = faster; 1 = sequential.
PARALLEL_WORKERS = int(os.getenv("PARALLEL_WORKERS", "3"))

## 3. Database Connections

Establish PostgreSQL and Neo4j connections. These are reused across all experiments.

In [3]:
pg_conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS
)
pg_conn.autocommit = True
print("PostgreSQL connected")

neo_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def neo_run(cypher, params=None):
    with neo_driver.session() as session:
        return session.run(cypher, params or {}).data()

print("Neo4j connected")

PostgreSQL connected
Neo4j connected


## 4. Load Test Data

Load the same test dataset for all experiments. Each row contains: id, complexity, question, expected_sql.

**Speed tip:** `PARALLEL_WORKERS=3` (Section 2) runs 3 tests in parallel. Set to 1 for sequential.

In [4]:
def load_test_data(csv_path: str = None) -> pd.DataFrame:
    """Load NL2SQL test cases from CSV. Same data used for all experiments."""
    candidates = [
        csv_path,
        CSV_PATH,
        os.path.join(os.path.dirname(os.path.dirname(os.path.abspath("."))), "test_data_set", "final_test_data.csv"),
        "test_data_set/final_test_data.csv",
        "../test_data_set/final_test_data.csv",
        "final_test_data.csv",
    ]
    for path in candidates:
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {len(df)} test cases from {path}")
            return df
    raise FileNotFoundError("Could not find final_test_data.csv. Set CSV_PATH or ensure test_data_set/final_test_data.csv exists.")

df_tests = load_test_data()
if MAX_TEST_CASES:
    df_tests = df_tests.head(MAX_TEST_CASES)
    print(f"Quick mode: running first {len(df_tests)} test cases (set MAX_TEST_CASES env to 0 or unset for full run)")
df_tests.head(5)

Loaded 75 test cases from /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/final_test_data.csv


,id,complexity,question,expected_sql
0,1,easy,List all products.,SELECT * FROM products;
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;
3,4,easy,Show all products in the 'Home & Kitchen' cate...,SELECT * FROM public.products WHERE products.c...
4,5,easy,Retrieve all customers who live in the state '...,SELECT * FROM customers WHERE state = 'CA';


## 5. Extract Schema from PostgreSQL

Fetch tables, columns, primary keys, and foreign keys from `information_schema` for KG construction.

In [5]:
def fetch_df(query, params=None):
    return pd.read_sql_query(query, pg_conn, params=params)

schemas = fetch_df("""
SELECT schema_name FROM information_schema.schemata
WHERE schema_name NOT IN ('pg_catalog','information_schema') ORDER BY schema_name;
""")

tables = fetch_df("""
SELECT table_schema, table_name FROM information_schema.tables
WHERE table_type='BASE TABLE' AND table_schema NOT IN ('pg_catalog','information_schema')
ORDER BY table_schema, table_name;
""")

columns = fetch_df("""
SELECT table_schema, table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema NOT IN ('pg_catalog','information_schema')
ORDER BY table_schema, table_name, ordinal_position;
""")

pk = fetch_df("""
SELECT tc.table_schema, tc.table_name, kcu.column_name
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
WHERE tc.constraint_type='PRIMARY KEY'
  AND tc.table_schema NOT IN ('pg_catalog','information_schema')
ORDER BY tc.table_schema, tc.table_name;
""")

fk = fetch_df("""
SELECT tc.table_schema, tc.table_name, kcu.column_name,
       ccu.table_schema AS ref_schema, ccu.table_name AS ref_table, ccu.column_name AS ref_column
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu ON tc.constraint_name = kcu.constraint_name
JOIN information_schema.constraint_column_usage ccu ON ccu.constraint_name = tc.constraint_name
WHERE tc.constraint_type='FOREIGN KEY'
  AND tc.table_schema NOT IN ('pg_catalog','information_schema')
ORDER BY tc.table_schema, tc.table_name;
""")

print(f"Schema: {len(schemas)} schemas, {len(tables)} tables, {len(columns)} columns, {len(pk)} PKs, {len(fk)} FKs")

Schema: 2 schemas, 5 tables, 39 columns, 5 PKs, 5 FKs


/var/folders/75/bzqf62356_l1pth2cm_p2vm00000gn/T/ipykernel_86948/4176936788.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, pg_conn, params=params)


## 6. Build Knowledge Graph from Schema

Create Neo4j graph: Schema → Table → Column, with PK and FK relationships. No semantic enrichment (descriptions/synonyms) at this stage.

In [6]:
def build_kg_from_schema(schemas, tables, columns, pk, fk):
    """Build Knowledge Graph: Schema → Table → Column with PK/FK relationships."""
    neo_run("MATCH (n) DETACH DELETE n")
    neo_run("CREATE CONSTRAINT schema_name IF NOT EXISTS FOR (s:Schema) REQUIRE s.name IS UNIQUE")
    neo_run("CREATE CONSTRAINT table_full IF NOT EXISTS FOR (t:Table) REQUIRE t.full_name IS UNIQUE")
    neo_run("CREATE CONSTRAINT column_full IF NOT EXISTS FOR (c:Column) REQUIRE c.full_name IS UNIQUE")

    for s in schemas["schema_name"].tolist():
        neo_run("MERGE (:Schema {name:$name})", {"name": s})

    for _, r in tables.iterrows():
        full = f"{r.table_schema}.{r.table_name}"
        neo_run("""
        MATCH (s:Schema {name:$schema}) MERGE (t:Table {full_name:$full})
        SET t.schema=$schema, t.name=$name MERGE (s)-[:HAS_TABLE]->(t)
        """, {"schema": r.table_schema, "name": r.table_name, "full": full})

    for _, r in columns.iterrows():
        t_full = f"{r.table_schema}.{r.table_name}"
        c_full = f"{t_full}.{r.column_name}"
        neo_run("""
        MATCH (t:Table {full_name:$t_full}) MERGE (c:Column {full_name:$c_full})
        SET c.schema=$schema, c.table=$table, c.name=$col, c.data_type=$dtype, c.nullable=$nullable
        MERGE (t)-[:HAS_COLUMN]->(c)
        """, {"t_full": t_full, "c_full": c_full, "schema": r.table_schema, "table": r.table_name,
              "col": r.column_name, "dtype": r.data_type, "nullable": (r.is_nullable == "YES")})

    for _, r in pk.iterrows():
        c_full = f"{r.table_schema}.{r.table_name}.{r.column_name}"
        t_full = f"{r.table_schema}.{r.table_name}"
        neo_run("MATCH (c:Column {full_name:$c}) MATCH (t:Table {full_name:$t}) SET c.is_pk = true MERGE (c)-[:PK_OF]->(t)",
                {"c": c_full, "t": t_full})

    for _, r in fk.iterrows():
        src = f"{r.table_schema}.{r.table_name}.{r.column_name}"
        dst = f"{r.ref_schema}.{r.ref_table}.{r.ref_column}"
        neo_run("MATCH (c1:Column {full_name:$src}) MATCH (c2:Column {full_name:$dst}) MERGE (c1)-[:FK_TO]->(c2)",
                {"src": src, "dst": dst})

build_kg_from_schema(schemas, tables, columns, pk, fk)
print("Knowledge Graph built ✅")

Knowledge Graph built ✅


## 7. Schema Context Builders

- **`build_schema_context_basic`** (E1): Full schema from KG without descriptions/synonyms — tables, columns, PK, FK only.
- **`apply_semantic_enrichment`** (E2/E3): Add table/column descriptions, synonyms, and example value hints to the KG.

In [7]:
def build_schema_context_basic(limit_tables=50):
    """E1: Full schema from KG without semantics (no descriptions/synonyms)."""
    data = neo_run("""
    MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)
    WITH t, collect({name:c.name, type:c.data_type, nullable:c.nullable, is_pk:coalesce(c.is_pk,false)}) AS cols
    RETURN t.full_name AS table, cols ORDER BY table LIMIT $limit
    """, {"limit": limit_tables})
    fk_data = neo_run("""
    MATCH (c1:Column)-[:FK_TO]->(c2:Column)
    RETURN c1.full_name AS from_col, c2.full_name AS to_col LIMIT 200
    """)
    lines = []
    for row in data:
        lines.append(f"TABLE {row['table']}")
        for col in row["cols"]:
            pk_tag = " PK" if col["is_pk"] else ""
            nn_tag = " NOT_NULL" if (col["nullable"] is False) else ""
            lines.append(f"  - {col['name']} ({col['type']}){pk_tag}{nn_tag}")
    if fk_data:
        lines.append("\nFOREIGN_KEYS")
        for r in fk_data:
            lines.append(f"  - {r['from_col']} -> {r['to_col']}")
    return "\n".join(lines)

def apply_semantic_enrichment():
    """E2/E3: Add descriptions, synonyms, example hints to KG (from Knowledge_Graph_optimization)."""
    queries = [
        "MATCH (t:Table {full_name:'public.customers'}) SET t.description = 'Customer master data: identity, contact, and location fields'",
        "MATCH (t:Table {full_name:'public.orders'}) SET t.description = 'Orders placed by customers with status, date, shipping address and total amount'",
        "MATCH (t:Table {full_name:'public.order_items'}) SET t.description = 'Line items for orders; each line references a product, with quantity and pricing'",
        "MATCH (t:Table {full_name:'public.products'}) SET t.description = 'Product catalog: name, category, price and inventory'",
        "MATCH (t:Table {full_name:'public.reviews'}) SET t.description = 'Customer reviews for products with rating and optional review text'",
        "MATCH (t:Table {full_name:'public.orders'})-[:HAS_COLUMN]->(c:Column {name:'status'}) SET c.description='Order lifecycle status (pending, processing, shipped, delivered)', c.synonyms=['state','order state','delivery status'], c.examples=['pending','processing','shipped','delivered']",
        "MATCH (t:Table {full_name:'public.orders'})-[:HAS_COLUMN]->(c:Column {name:'total_amount'}) SET c.description='Total amount for the order (numeric), often sum of order_items.subtotal', c.synonyms=['revenue','total price','order amount','amount paid']",
        "MATCH (t:Table {full_name:'public.order_items'})-[:HAS_COLUMN]->(c:Column {name:'subtotal'}) SET c.description='Line subtotal = quantity * unit_price', c.synonyms=['line total','item total']",
        "MATCH (t:Table {full_name:'public.reviews'})-[:HAS_COLUMN]->(c:Column {name:'rating'}) SET c.description='Integer rating score (typically 1-5)', c.synonyms=['stars','score','review rating'], c.examples=[1,2,3,4,5]",
        "MATCH (t:Table {full_name:'public.products'})-[:HAS_COLUMN]->(c:Column {name:'stock_quantity'}) SET c.description='Inventory units available', c.synonyms=['stock','inventory','qty in stock','availability']",
    ]
    for q in queries:
        neo_run(q)
    print("Semantic enrichment applied ✅")

## 8. Shared Utilities

Functions used across experiments: SQL normalization, execution, result comparison, JSON serialization.

In [8]:
def normalize_sql(sql: str) -> str:
    if not sql: return ""
    sql = sql.strip().strip("`").split(";")[0].strip() + ";"
    sql = sqlparse.format(sql, keyword_case="upper", strip_comments=True, reindent=False)
    return re.sub(r"\s+", " ", sql).strip()

def json_safe(obj):
    if isinstance(obj, (datetime.datetime, datetime.date)): return obj.isoformat()
    if isinstance(obj, decimal.Decimal): return float(obj)
    if isinstance(obj, uuid.UUID): return str(obj)
    if isinstance(obj, (bytes, bytearray)): return obj.decode("utf-8", errors="replace")
    return str(obj)

def dumps_safe(obj): return json.dumps(obj, ensure_ascii=False, default=json_safe)

def rows_to_jsonable(cols, rows):
    if cols is None or rows is None: return None
    return [{cols[i]: json_safe(row[i]) for i in range(len(cols))} for row in rows]

def rows_fingerprint(cols, rows):
    if rows is None: return None
    m = hashlib.sha256()
    m.update(repr(cols).encode("utf-8")); m.update(repr(rows).encode("utf-8"))
    return m.hexdigest()

_tl = threading.local()
def _get_pg_conn():
    """Per-thread conn when PARALLEL_WORKERS>1; else global pg_conn."""
    if PARALLEL_WORKERS and PARALLEL_WORKERS > 1:
        if not getattr(_tl, 'conn', None) or _tl.conn.closed:
            _tl.conn = psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS)
            _tl.conn.autocommit = True
        return _tl.conn
    return pg_conn

def try_execute(sql: str, fetch_rows: int = FETCH_ROWS):
    conn = _get_pg_conn()
    cur = None
    try:
        cur = conn.cursor()
        cur.execute(sql)
        cols, rows = None, None
        if cur.description:
            cols = [d[0] for d in cur.description]
            rows = cur.fetchmany(fetch_rows)
        cur.close()
        return True, cols, rows, None
    except Exception as e:
        try:
            if cur: cur.close()
        except: pass
        return False, None, None, str(e)

def result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err, gen_ok, gen_cols, gen_rows, gen_err, exact_match, result_match):
    if gen_err and str(gen_err).strip().lower() == "empty sql": return "empty sql"
    if not gen_ok: return f"generated sql execution failed: {gen_err}"
    if not exp_ok: return f"expected sql execution failed: {exp_err}"
    if (exp_cols is None) != (gen_cols is None): return "one query returned rows, the other did not"
    if exp_cols != gen_cols: return "column mismatch"
    if len(exp_rows or []) != len(gen_rows or []): return "row count mismatch (sampled)"
    if result_match: return "ok"
    if exact_match: return "sql matches but sample differs"
    return "sample rows mismatch"

def run_parallel(df_tests: pd.DataFrame, process_fn, desc: str = "E"):
    """Run process_fn(i, row) for each row. process_fn returns result dict. Uses threads if PARALLEL_WORKERS>1."""
    rows = [(i, r) for i, (_, r) in enumerate(df_tests.iterrows())]
    if PARALLEL_WORKERS and PARALLEL_WORKERS > 1:
        results = [None] * len(rows)
        with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
            futures = {ex.submit(process_fn, i, r): i for i, r in rows}
            for f in tqdm(as_completed(futures), total=len(futures), desc=desc):
                results[futures[f]] = f.result()
        return pd.DataFrame([r for r in results if r is not None])
    return pd.DataFrame([process_fn(i, r) for i, r in tqdm(rows, desc=desc)])

## 9. Experiment 1: Baseline One-Shot (KG schema, no semantics)

**Objective:** Establish baseline NL2SQL with minimal controls.

**Config:** Model gemma3:4b | Schema: full schema from KG (tables/columns/FKs) **without** descriptions/synonyms | Prompting: single-shot SQL generation | Execution: run once, no repair.

In [9]:
SYSTEM_RULES = """You are a senior data engineer. Return ONLY valid PostgreSQL SQL.
No explanation. No markdown. No comments. Use schema-qualified names exactly as in the schema.
Return exactly ONE SQL statement."""

def build_prompt_e1(nl_question: str, schema_ctx: str) -> str:
    return f"""{SYSTEM_RULES}

SCHEMA_CONTEXT:
{schema_ctx}

QUESTION:
{nl_question}

SQL:
"""

def ollama_generate(prompt: str, temperature: float = TEMPERATURE, model: str = None) -> str:
    m = model or OLLAMA_MODEL
    resp = ollama.generate(model=m, prompt=prompt, options={"temperature": float(temperature)}, stream=False)
    return (resp.get("response") or "").strip()

def run_experiment_1(df_tests: pd.DataFrame) -> pd.DataFrame:
    """E1: Baseline one-shot with KG schema, no semantics."""
    schema_context = build_schema_context_basic(limit_tables=200)
    def process(i, r):
        qid, complexity = int(r["id"]), str(r["complexity"])
        question, expected_sql_raw = str(r["question"]), str(r["expected_sql"]).strip()
        expected_sql = normalize_sql(expected_sql_raw)
        prompt = build_prompt_e1(question, schema_context)
        t0 = time.time()
        try:
            raw = ollama_generate(prompt)
            generated_sql = normalize_sql(raw)
            llm_error = ""
        except Exception as e:
            generated_sql, llm_error = "", str(e)
        llm_latency_ms = int((time.time() - t0) * 1000)
        exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)
        gen_ok, gen_cols, gen_rows, gen_err = (try_execute(generated_sql, fetch_rows=FETCH_ROWS) if generated_sql else (False, None, None, "Empty SQL"))
        exact_match = (expected_sql == generated_sql) if generated_sql else False
        result_match = (rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)) if (exp_ok and gen_ok) else False
        hint = result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err, gen_ok, gen_cols, gen_rows, gen_err, exact_match, result_match)
        generated_ok = bool(gen_ok and (result_match or exact_match))
        return {"id": qid, "complexity": complexity, "question": question, "expected_sql": expected_sql, "generated_sql": generated_sql,
            "llm_latency_ms": llm_latency_ms, "llm_error": llm_error, "expected_exec_ok": exp_ok, "generated_exec_ok": gen_ok,
            "generated_exec_error": gen_err or "", "exact_match": exact_match, "result_match_sample": result_match, "result_diff_hint": hint, "generated_ok": generated_ok,
            "expected_result_sample_json": dumps_safe(rows_to_jsonable(exp_cols, exp_rows)) or "", "generated_result_sample_json": dumps_safe(rows_to_jsonable(gen_cols, gen_rows)) or ""}
    return run_parallel(df_tests, process, "E1")

## 10. Experiment 2: Semantic KG + Multi-Stage LLM (Schema Linking)

**Objective:** Reduce hallucinations via schema linking and multi-stage pipeline.

**Config:** Model gemma3:4b | KG: table/column descriptions + synonyms + example value hints (from Knowledge_Graph_optimization) | Pipeline: table selection → KG retrieval → column selection → minimal schema → final SQL | Explicit rules for table/column selection and SELECT *.

In [10]:
FULL_SCHEMA_TEXT = build_schema_context_basic(200)

def extract_json(text: str):
    try: return json.loads(text)
    except:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m: raise ValueError("No JSON found")
        return json.loads(m.group(0))

TABLE_SELECTOR_PROMPT = """You are a PostgreSQL schema expert.

Return STRICT JSON only:
{
  "tables": ["public.table1","public.table2",...],
  "reason": {"public.table1":"...", "public.table2":"..."}
}

Rules:
- Choose ONLY tables required to answer the question.
- If mentions rating/reviews -> reviews/products.
- If mentions order total/revenue/status/date -> orders (and customers for customer fields).
- If mentions quantity/subtotal/unit_price -> order_items (+ orders/products as needed).
- If mentions customer info -> customers.
- Do NOT invent tables.
- JSON only. No markdown."""

def llm_select_tables(question: str, model: str = None):
    prompt = f"{TABLE_SELECTOR_PROMPT}\n\nSCHEMA:\n{FULL_SCHEMA_TEXT}\n\nQUESTION:\n{question}\n\nJSON:\n"
    return extract_json(ollama_generate(prompt, temperature=0.0, model=model))

def kg_fetch_tables_with_metadata(tables: list):
    cols = neo_run("""
    MATCH (t:Table)-[:HAS_COLUMN]->(c:Column) WHERE t.full_name IN $tables
    RETURN t.full_name AS table, c.name AS col, coalesce(c.data_type,'') AS data_type,
           coalesce(c.nullable,true) AS nullable, coalesce(c.is_pk,false) AS is_pk,
           coalesce(c.description,'') AS description, coalesce(c.synonyms,[]) AS synonyms, coalesce(c.examples,[]) AS examples
    ORDER BY table, col
    """, {"tables": tables})
    joins = neo_run("""
    MATCH (c1:Column)-[:FK_TO]->(c2:Column) MATCH (t1:Table)-[:HAS_COLUMN]->(c1) MATCH (t2:Table)-[:HAS_COLUMN]->(c2)
    WHERE t1.full_name IN $tables AND t2.full_name IN $tables
    RETURN t1.full_name AS from_table, c1.name AS from_col, t2.full_name AS to_table, c2.name AS to_col
    """, {"tables": tables})
    seen = set()
    uniq = [j for j in joins if (k := (j["from_table"],j["from_col"],j["to_table"],j["to_col"])) not in seen and not seen.add(k)]
    return cols, uniq

def render_table_schemas(cols_rows, joins_rows):
    by_table = {}
    for r in cols_rows: by_table.setdefault(r["table"], []).append(r)
    lines = []
    for t, cols in by_table.items():
        lines.append(f"TABLE {t}")
        for c in cols:
            pk = " PK" if c["is_pk"] else ""
            nn = " NOT_NULL" if (c["nullable"] is False) else ""
            desc = f" -- {c['description']}" if c.get("description") else ""
            lines.append(f"  - {c['col']} ({c['data_type']}){pk}{nn}{desc}")
            if c.get("synonyms"): lines.append(f"    synonyms: {', '.join(c['synonyms'])}")
            if c.get("examples"): lines.append(f"    examples: {c['examples']}")
    if joins_rows:
        lines.append("\nALLOWED_JOINS (use only these):")
        for j in joins_rows: lines.append(f"  - {j['from_table']}.{j['from_col']} = {j['to_table']}.{j['to_col']}")
    return "\n".join(lines)

COLUMN_SELECTOR_PROMPT = """You select the minimal columns needed to write correct SQL.

Return STRICT JSON only:
{
  "columns": {"public.table1": ["colA","colB",...], ...},
  "needs_aggregation": true/false,
  "group_by": ["public.t.col",...],
  "order_by": [{"column":"public.t.col","direction":"ASC|DESC"}],
  "limit": null|number,
  "filters": [{"column":"public.t.col","op":"=","value_hint":"..."}]
}

Rules:
- Use only columns present in TABLE_SCHEMAS below.
- Choose only columns needed for SELECT/JOIN/WHERE/GROUP BY/ORDER BY.
- SELECT * rule: If question contains "list all"/"show all"/"find all"/"get all" and user did NOT ask for specific fields, set main table columns to ["*"].
- If aggregation is requested, do NOT use "*".
- JSON only."""

def llm_select_columns(question: str, table_schema_text: str, model: str = None):
    prompt = f"{COLUMN_SELECTOR_PROMPT}\n\nTABLE_SCHEMAS:\n{table_schema_text}\n\nQUESTION:\n{question}\n\nJSON:\n"
    return extract_json(ollama_generate(prompt, temperature=0.0, model=model))

def kg_fetch_minimal_schema(selected_tables, selected_columns_map, joins_rows):
    join_cols = {}
    for j in joins_rows:
        join_cols.setdefault(j["from_table"], set()).add(j["from_col"])
        join_cols.setdefault(j["to_table"], set()).add(j["to_col"])
    cols_rows = neo_run("""
    MATCH (t:Table)-[:HAS_COLUMN]->(c:Column) WHERE t.full_name IN $tables
    RETURN t.full_name AS table, c.name AS col, coalesce(c.data_type,'') AS data_type,
           coalesce(c.nullable,true) AS nullable, coalesce(c.is_pk,false) AS is_pk, coalesce(c.description,'') AS description
    """, {"tables": selected_tables})
    by_table = {}
    for r in cols_rows: by_table.setdefault(r["table"], []).append(r)
    filtered = {}
    for t in selected_tables:
        requested = set(selected_columns_map.get(t, []))
        star = "*" in requested
        if star: requested = set()
        for c in by_table.get(t, []):
            if c["is_pk"]: requested.add(c["col"])
        for jc in join_cols.get(t, set()): requested.add(jc)
        filtered[t] = {"star": star, "cols": [c for c in by_table.get(t, []) if c["col"] in requested]}
    return filtered

def render_minimal_schema(filtered, joins_rows):
    lines = []
    for t, info in filtered.items():
        lines.append(f"TABLE {t}")
        if info["star"]: lines.append("  - * (all columns)")
        for c in info["cols"]:
            pk = " PK" if c["is_pk"] else ""
            nn = " NOT_NULL" if (c["nullable"] is False) else ""
            desc = f" -- {c['description']}" if c.get("description") else ""
            lines.append(f"  - {c['col']} ({c['data_type']}){pk}{nn}{desc}")
    if joins_rows:
        lines.append("\nALLOWED_JOINS (use only these):")
        for j in joins_rows: lines.append(f"  - {j['from_table']}.{j['from_col']} = {j['to_table']}.{j['to_col']}")
    return "\n".join(lines)

SQL_GEN_PROMPT = """You are a senior PostgreSQL data engineer.

Generate exactly ONE valid PostgreSQL SQL statement.

STRICT RULES:
1) Return ONLY SQL. No explanation/markdown/comments.
2) Use schema-qualified names exactly as provided (e.g., public.orders).
3) Use ONLY tables/columns listed in SCHEMA below.
4) Use ONLY joins listed under ALLOWED_JOINS.
5) SELECT * rule: If question contains "list all"/"show all"/"find all"/"get all" and user did NOT ask specific fields, use SELECT * for main table(s). Otherwise select only required columns.
6) If aggregation is requested, do NOT use SELECT *.
7) End with semicolon."""

def llm_generate_sql(question: str, minimal_schema_text: str, model: str = None, few_shot_examples: str = None):
    examples = f"\nFEW-SHOT EXAMPLES:\n{few_shot_examples}\n\n" if few_shot_examples else ""
    prompt = f"{SQL_GEN_PROMPT}{examples}SCHEMA:\n{minimal_schema_text}\n\nQUESTION:\n{question}\n\nSQL:\n"
    return normalize_sql(ollama_generate(prompt, temperature=TEMPERATURE, model=model))

def nl2sql_pipeline_e2(question: str, model: str = None, few_shot_examples: str = None):
    tsel = llm_select_tables(question, model=model)
    tables = tsel.get("tables", []) or ["public.customers","public.orders","public.order_items","public.products","public.reviews"]
    cols_rows, joins_rows = kg_fetch_tables_with_metadata(tables)
    table_schema_text = render_table_schemas(cols_rows, joins_rows)
    csel = llm_select_columns(question, table_schema_text, model=model)
    columns_map = csel.get("columns", {}) or {}
    filtered = kg_fetch_minimal_schema(tables, columns_map, joins_rows)
    minimal_schema_text = render_minimal_schema(filtered, joins_rows)
    sql = llm_generate_sql(question, minimal_schema_text, model=model, few_shot_examples=few_shot_examples)
    return {"tables_selected": tables, "minimal_schema_text": minimal_schema_text, "sql": sql}

In [11]:
def run_experiment_2(df_tests: pd.DataFrame) -> pd.DataFrame:
    """E2: Semantic KG + multi-stage LLM pipeline."""
    apply_semantic_enrichment()
    global FULL_SCHEMA_TEXT
    FULL_SCHEMA_TEXT = build_schema_context_basic(200)
    def process(i, r):
        qid, complexity = int(r["id"]), str(r["complexity"])
        question, expected_sql_raw = str(r["question"]), str(r["expected_sql"]).strip()
        expected_sql = normalize_sql(expected_sql_raw)
        t0 = time.time()
        try:
            pipe = nl2sql_pipeline_e2(question)
            generated_sql = pipe["sql"]
            llm_error = ""
        except Exception as e:
            generated_sql, llm_error = "", str(e)
        llm_latency_ms = int((time.time() - t0) * 1000)
        exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)
        gen_ok, gen_cols, gen_rows, gen_err = (try_execute(generated_sql, fetch_rows=FETCH_ROWS) if generated_sql else (False, None, None, "Empty SQL"))
        exact_match = (expected_sql == generated_sql) if generated_sql else False
        result_match = (rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)) if (exp_ok and gen_ok) else False
        hint = result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err, gen_ok, gen_cols, gen_rows, gen_err, exact_match, result_match)
        generated_ok = bool(gen_ok and (result_match or exact_match))
        return {"id": qid, "complexity": complexity, "question": question, "expected_sql": expected_sql, "generated_sql": generated_sql,
            "llm_latency_ms": llm_latency_ms, "llm_error": llm_error, "expected_exec_ok": exp_ok, "generated_exec_ok": gen_ok,
            "generated_exec_error": gen_err or "", "exact_match": exact_match, "result_match_sample": result_match, "result_diff_hint": hint, "generated_ok": generated_ok,
            "expected_result_sample_json": dumps_safe(rows_to_jsonable(exp_cols, exp_rows)) or "", "generated_result_sample_json": dumps_safe(rows_to_jsonable(gen_cols, gen_rows)) or ""}
    return run_parallel(df_tests, process, "E2")

## 11. Experiment 3: Execution-Guided Retry (Multi-Shot Repair)

**Objective:** Improve robustness via correction loop when SQL fails.

**Config:** Model gemma3:4b | Pipeline: same as E2 (with KG optimization prompts) | Retry: one additional generation if SQL fails; repair prompt includes failed SQL + Postgres error + minimal schema context.

In [12]:
REPAIR_PROMPT = """You are a PostgreSQL expert. The following SQL FAILED with a Postgres error.
Fix the SQL. Return ONLY the corrected SQL. No explanation. Use schema-qualified names from SCHEMA.
End with semicolon."""

def llm_repair_sql(question: str, minimal_schema_text: str, failed_sql: str, db_error: str, model: str = None, example_context: str = None) -> str:
    examples = f"\nREFERENCE EXAMPLES:\n{example_context}\n\n" if example_context else ""
    prompt = f"{REPAIR_PROMPT}{examples}ORIGINAL QUESTION:\n{question}\n\nSCHEMA:\n{minimal_schema_text}\n\nFAILED SQL:\n{failed_sql}\n\nPOSTGRES ERROR:\n{db_error}\n\nCORRECTED SQL:\n"
    return normalize_sql(ollama_generate(prompt, temperature=TEMPERATURE, model=model))

def generate_sql_with_retry(question: str, model: str = None, few_shot_examples: str = None, example_context: str = None):
    """E3/E5: E2 pipeline + one retry on execution failure."""
    pipe = nl2sql_pipeline_e2(question, model=model, few_shot_examples=few_shot_examples)
    sql1 = pipe["sql"]
    minimal_schema_text = pipe["minimal_schema_text"]
    ok1, cols1, rows1, err1 = try_execute(sql1, fetch_rows=FETCH_ROWS) if sql1 else (False, None, None, "Empty SQL")
    if ok1:
        return sql1, pipe
    sql2 = llm_repair_sql(question, minimal_schema_text, failed_sql=sql1, db_error=err1 or "", model=model, example_context=example_context)
    return sql2, pipe

In [13]:
def run_experiment_3(df_tests: pd.DataFrame) -> pd.DataFrame:
    """E3: Semantic KG + multi-stage + execution-guided retry."""
    apply_semantic_enrichment()
    global FULL_SCHEMA_TEXT
    FULL_SCHEMA_TEXT = build_schema_context_basic(200)
    def process(i, r):
        qid, complexity = int(r["id"]), str(r["complexity"])
        question, expected_sql_raw = str(r["question"]), str(r["expected_sql"]).strip()
        expected_sql = normalize_sql(expected_sql_raw)
        t0 = time.time()
        try:
            generated_sql, pipe = generate_sql_with_retry(question)
            llm_error = ""
        except Exception as e:
            generated_sql, llm_error = "", str(e)
        llm_latency_ms = int((time.time() - t0) * 1000)
        exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)
        gen_ok, gen_cols, gen_rows, gen_err = (try_execute(generated_sql, fetch_rows=FETCH_ROWS) if generated_sql else (False, None, None, "Empty SQL"))
        exact_match = (expected_sql == generated_sql) if generated_sql else False
        result_match = (rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)) if (exp_ok and gen_ok) else False
        hint = result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err, gen_ok, gen_cols, gen_rows, gen_err, exact_match, result_match)
        generated_ok = bool(gen_ok and (result_match or exact_match))
        return {"id": qid, "complexity": complexity, "question": question, "expected_sql": expected_sql, "generated_sql": generated_sql,
            "llm_latency_ms": llm_latency_ms, "llm_error": llm_error, "expected_exec_ok": exp_ok, "generated_exec_ok": gen_ok,
            "generated_exec_error": gen_err or "", "exact_match": exact_match, "result_match_sample": result_match, "result_diff_hint": hint, "generated_ok": generated_ok,
            "expected_result_sample_json": dumps_safe(rows_to_jsonable(exp_cols, exp_rows)) or "", "generated_result_sample_json": dumps_safe(rows_to_jsonable(gen_cols, gen_rows)) or ""}
    return run_parallel(df_tests, process, "E3")

## 12. Experiment 4: SQL-Specialized Model Upgrade

**Objective:** Evaluate defog/sqlcoder-7b-2 on correctness. Pipeline: same as E2/E3 | Retry: enabled | Expectation: better joins, aggregations; higher latency.

In [14]:
def run_experiment_4(df_tests: pd.DataFrame, retry_enabled: bool = True) -> pd.DataFrame:
    """E4: SQL-specialized model (sqlcoder) with E2/E3 pipeline, retry enabled."""
    apply_semantic_enrichment()
    global FULL_SCHEMA_TEXT
    FULL_SCHEMA_TEXT = build_schema_context_basic(200)
    model = OLLAMA_MODEL_E4
    def process(i, r):
        qid, complexity = int(r["id"]), str(r["complexity"])
        question, expected_sql_raw = str(r["question"]), str(r["expected_sql"]).strip()
        expected_sql = normalize_sql(expected_sql_raw)
        t0 = time.time()
        try:
            if retry_enabled:
                generated_sql, _ = generate_sql_with_retry(question, model=model)
            else:
                pipe = nl2sql_pipeline_e2(question, model=model)
                generated_sql = pipe["sql"]
            llm_error = ""
        except Exception as e:
            generated_sql, llm_error = "", str(e)
        llm_latency_ms = int((time.time() - t0) * 1000)
        exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)
        gen_ok, gen_cols, gen_rows, gen_err = (try_execute(generated_sql, fetch_rows=FETCH_ROWS) if generated_sql else (False, None, None, "Empty SQL"))
        exact_match = (expected_sql == generated_sql) if generated_sql else False
        result_match = (rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)) if (exp_ok and gen_ok) else False
        hint = result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err, gen_ok, gen_cols, gen_rows, gen_err, exact_match, result_match)
        generated_ok = bool(gen_ok and (result_match or exact_match))
        return {"id": qid, "complexity": complexity, "question": question, "expected_sql": expected_sql, "generated_sql": generated_sql,
            "llm_latency_ms": llm_latency_ms, "llm_error": llm_error, "expected_exec_ok": exp_ok, "generated_exec_ok": gen_ok,
            "generated_exec_error": gen_err or "", "exact_match": exact_match, "result_match_sample": result_match, "result_diff_hint": hint, "generated_ok": generated_ok,
            "expected_result_sample_json": dumps_safe(rows_to_jsonable(exp_cols, exp_rows)) or "", "generated_result_sample_json": dumps_safe(rows_to_jsonable(gen_cols, gen_rows)) or ""}
    return run_parallel(df_tests, process, "E4")

## 13. Experiment 5: Example-Augmented KG Grounding (Few-Shot Retrieval)

**Objective:** Improve medium/hard queries via validated SQL examples in the Knowledge Graph.

**Config:** Model gemma3:4b | Base pipeline: E2/E3 schema-grounded KG RAG | KG: 20 Query Examples (10 medium + 10 hard) with NL, gold SQL, tables, join paths, complexity | Top-K (2–3) retrieval by schema overlap + structural similarity | Few-shot injection + retry with example context.

In [15]:
def extract_tables_from_sql(sql: str) -> set:
    """Extract table names from SQL (FROM/JOIN)."""
    if not sql: return set()
    tables = set()
    for m in re.finditer(r"(?:FROM|JOIN)\s+(?:([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)|([a-z_][a-z0-9_]*))", sql, re.I):
        if m.group(1): tables.add(f"{m.group(1)}.{m.group(2)}")
        elif m.group(3): tables.add(f"public.{m.group(3)}")
    return tables

def extract_sql_structure(sql: str) -> dict:
    """Structural features: has_join, has_group_by, has_subquery, has_having, has_window."""
    s = (sql or "").upper()
    return {
        "has_join": " JOIN " in s,
        "has_group_by": " GROUP BY " in s,
        "has_subquery": "(" in s and ("SELECT " in s or "EXISTS" in s),
        "has_having": " HAVING " in s,
        "has_window": " OVER " in s or "PARTITION BY" in s,
    }

# Seed 20 examples (10 medium + 10 hard) from test data - exclude current query at runtime
def build_query_examples(df_tests: pd.DataFrame, n_medium: int = 10, n_hard: int = 10):
    medium = df_tests[df_tests["complexity"] == "medium"].head(n_medium)
    hard = df_tests[df_tests["complexity"] == "hard"].head(n_hard)
    examples = []
    for _, r in pd.concat([medium, hard]).iterrows():
        sql = normalize_sql(str(r["expected_sql"]))
        tables = extract_tables_from_sql(sql)
        structure = extract_sql_structure(sql)
        examples.append({
            "question": str(r["question"]),
            "gold_sql": sql,
            "tables_used": tables,
            "join_paths": list(tables),  # simplified; could parse ON clauses
            "complexity": str(r["complexity"]),
            "structure": structure,
        })
    return examples

QUERY_EXAMPLES = None  # populated when run_experiment_5 is called

def retrieve_similar_examples(question: str, selected_tables: list, complexity: str, examples: list, top_k: int = 3, exclude_question: str = None):
    """Retrieve top-K examples by schema overlap + structural similarity. Exclude current query to avoid leakage."""
    if exclude_question:
        examples = [ex for ex in examples if ex["question"].strip().lower() != exclude_question.strip().lower()]
    q_words = set(re.findall(r"\w+", question.lower()))
    q_struct_approx = {
        "has_join": any(w in q_words for w in ["join", "together", "with", "each", "for each"]),
        "has_group_by": any(w in q_words for w in ["each", "per", "count", "sum", "avg", "total", "group"]),
        "has_subquery": any(w in q_words for w in ["never", "not", "without", "above", "below", "average"]),
        "has_having": any(w in q_words for w in ["only", "at least", "more than", "greater"]),
        "has_window": "window" in q_words or "rank" in q_words,
    }
    scored = []
    sel_set = set(t.replace("public.", "") for t in selected_tables)
    for ex in examples:
        if ex["complexity"] != complexity and (complexity == "medium" or complexity == "hard"):
            pass  # prefer same complexity
        ex_set = set(t.replace("public.", "") for t in ex["tables_used"])
        overlap = len(sel_set & ex_set) / max(1, len(sel_set | ex_set))
        struct_score = sum(1 for k in q_struct_approx if q_struct_approx[k] and ex["structure"].get(k, False)) / 5.0
        score = 0.6 * overlap + 0.4 * (0.5 + 0.5 * struct_score)
        if ex["complexity"] == complexity:
            score += 0.2
        scored.append((score, ex))
    scored.sort(key=lambda x: -x[0])
    return [ex for _, ex in scored[:top_k]]

def format_few_shot_examples(examples: list) -> str:
    if not examples: return ""
    lines = []
    for i, ex in enumerate(examples, 1):
        lines.append(f"Example {i}:\nQ: {ex['question']}\nA: {ex['gold_sql']}\n")
    return "\n".join(lines)

def run_experiment_5(df_tests: pd.DataFrame) -> pd.DataFrame:
    """E5: Example-augmented KG with few-shot retrieval + retry with example context."""
    global QUERY_EXAMPLES
    QUERY_EXAMPLES = build_query_examples(df_tests, n_medium=10, n_hard=10)
    apply_semantic_enrichment()
    global FULL_SCHEMA_TEXT
    FULL_SCHEMA_TEXT = build_schema_context_basic(200)
    def process(i, r):
        qid, complexity = int(r["id"]), str(r["complexity"])
        question, expected_sql_raw = str(r["question"]), str(r["expected_sql"]).strip()
        expected_sql = normalize_sql(expected_sql_raw)
        t0 = time.time()
        try:
            tsel = llm_select_tables(question)
            tables = tsel.get("tables", []) or ["public.customers","public.orders","public.order_items","public.products","public.reviews"]
            examples = retrieve_similar_examples(question, tables, complexity, QUERY_EXAMPLES, top_k=3, exclude_question=question)
            few_shot = format_few_shot_examples(examples)
            generated_sql, pipe = generate_sql_with_retry(question, few_shot_examples=few_shot, example_context=few_shot)
            llm_error = ""
        except Exception as e:
            generated_sql, llm_error = "", str(e)
        llm_latency_ms = int((time.time() - t0) * 1000)
        exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)
        gen_ok, gen_cols, gen_rows, gen_err = (try_execute(generated_sql, fetch_rows=FETCH_ROWS) if generated_sql else (False, None, None, "Empty SQL"))
        exact_match = (expected_sql == generated_sql) if generated_sql else False
        result_match = (rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)) if (exp_ok and gen_ok) else False
        hint = result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err, gen_ok, gen_cols, gen_rows, gen_err, exact_match, result_match)
        generated_ok = bool(gen_ok and (result_match or exact_match))
        return {"id": qid, "complexity": complexity, "question": question, "expected_sql": expected_sql, "generated_sql": generated_sql,
            "llm_latency_ms": llm_latency_ms, "llm_error": llm_error, "expected_exec_ok": exp_ok, "generated_exec_ok": gen_ok,
            "generated_exec_error": gen_err or "", "exact_match": exact_match, "result_match_sample": result_match, "result_diff_hint": hint, "generated_ok": generated_ok,
            "expected_result_sample_json": dumps_safe(rows_to_jsonable(exp_cols, exp_rows)) or "", "generated_result_sample_json": dumps_safe(rows_to_jsonable(gen_cols, gen_rows)) or ""}
    return run_parallel(df_tests, process, "E5")

## 13. Run Experiments and Save Results

Execute experiments and save results to `test_data_set/nl2sql_experiment_{N}_results.csv` and summary to `nl2sql_experiment_{N}_summary.csv`.

In [16]:
def save_experiment_results(df_report: pd.DataFrame, experiment_num: int, results_dir: str = None):
    """Save detailed results and summary per experiment."""
    results_dir = results_dir or RESULTS_DIR
    os.makedirs(results_dir, exist_ok=True)
    details_path = os.path.join(results_dir, f"nl2sql_experiment_{experiment_num}_results.csv")
    summary_path = os.path.join(results_dir, f"nl2sql_experiment_{experiment_num}_summary.csv")
    df_report.to_csv(details_path, index=False)
    summary = df_report.groupby("complexity").agg(
        total=("id", "count"),
        exact_acc=("exact_match", "mean"),
        exec_ok_rate=("generated_exec_ok", "mean"),
        result_acc=("result_match_sample", "mean"),
        pass_rate=("generated_ok", "mean"),
        avg_llm_ms=("llm_latency_ms", "mean"),
    ).reset_index()
    order = ["easy", "medium", "hard"]
    if "complexity" in summary.columns:
        summary["complexity"] = pd.Categorical(summary["complexity"], categories=order, ordered=True)
        summary = summary.sort_values("complexity")
    summary.to_csv(summary_path, index=False)
    print(f"Saved: {details_path}")
    print(f"Saved: {summary_path}")
    return summary

In [17]:
# Ensure KG is built before experiments
build_kg_from_schema(schemas, tables, columns, pk, fk)

# Run Experiment 1 (Baseline)
df_e1 = run_experiment_1(df_tests)
summary_e1 = save_experiment_results(df_e1, 1)
display(df_e1.head(10))
display(summary_e1)

E1: 100%|███████████████████████████████████████| 75/75 [02:32<00:00,  2.03s/it]

Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_1_results.csv
Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_1_summary.csv


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,expected_exec_ok,generated_exec_ok,generated_exec_error,exact_match,result_match_sample,result_diff_hint,generated_ok,expected_result_sample_json,generated_result_sample_json
0,1,easy,List all products.,SELECT * FROM products;,SELECT public.products.* FROM public.products;,7032,,True,True,,False,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,SELECT products.product_name FROM public.produ...,6496,,True,True,,False,False,column mismatch,False,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_name"": ""Laptop Pro 15\""""}, {""produc..."
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,SELECT DISTINCT public.products.* FROM public....,5637,,True,True,,False,False,sample rows mismatch,False,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""6866d464-13e2-476e-bcd5-5686d..."
3,4,easy,Show all products in the 'Home & Kitchen' cate...,SELECT * FROM public.products WHERE products.c...,"SELECT products.product_name, products.price F...",2708,,True,True,,False,False,column mismatch,False,"[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738...","[{""product_name"": ""Stand Mixer"", ""price"": 299...."
4,5,easy,Retrieve all customers who live in the state '...,SELECT * FROM customers WHERE state = 'CA';,SELECT * FROM public.customers WHERE state = '...,2520,,True,True,,False,True,ok,True,"[{""customer_id"": ""6074865c-206a-44a4-9826-84dc...","[{""customer_id"": ""6074865c-206a-44a4-9826-84dc..."
5,6,easy,Get all orders with status 'delivered'.,SELECT * FROM orders WHERE status = 'delivered';,SELECT * FROM public.orders WHERE status = 'de...,2659,,True,True,,False,True,ok,True,"[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23...","[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23..."
6,7,easy,List all order items for the laptop product wh...,SELECT oi.* FROM order_items oi JOIN products ...,SELECT oi.* FROM public.order_items AS oi JOIN...,3072,,True,True,,False,True,ok,True,"[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b...","[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b..."
7,8,easy,Show all product reviews with a rating of 5.,SELECT * FROM reviews WHERE rating = 5;,SELECT public.reviews.* FROM public.reviews WH...,3325,,True,True,,False,True,ok,True,"[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874...","[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874..."
8,9,easy,Get the distinct product categories available ...,SELECT DISTINCT category FROM products;,SELECT DISTINCT public.products.category FROM ...,3165,,True,True,,False,True,ok,True,"[{""category"": ""Home & Kitchen""}, {""category"": ...","[{""category"": ""Home & Kitchen""}, {""category"": ..."
9,10,easy,Count the total number of orders.,SELECT COUNT(*) FROM orders;,SELECT count(*) FROM public.orders;,1779,,True,True,,False,True,ok,True,"[{""count"": ""15""}]","[{""count"": ""15""}]"


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.04,1.00,0.72,0.72,3801.16
2,medium,25,0.00,1.00,0.28,0.28,5719.00
1,hard,25,0.00,0.96,0.08,0.08,8274.00


In [18]:
# Run Experiment 2 (Semantic KG + Multi-Stage)
# Rebuild KG and apply semantic enrichment (run_experiment_2 does this)
df_e2 = run_experiment_2(df_tests)
summary_e2 = save_experiment_results(df_e2, 2)
display(df_e2.head(10))
display(summary_e2)

Semantic enrichment applied ✅


E2: 100%|███████████████████████████████████████| 75/75 [13:18<00:00, 10.64s/it]

Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_2_results.csv
Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_2_summary.csv


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,expected_exec_ok,generated_exec_ok,generated_exec_error,exact_match,result_match_sample,result_diff_hint,generated_ok,expected_result_sample_json,generated_result_sample_json
0,1,easy,List all products.,SELECT * FROM products;,SELECT * FROM public.products;,20958,,True,True,,False,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,SELECT * FROM public.products WHERE category =...,21586,,True,True,,True,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,SELECT * FROM public.products WHERE price > 100;,20411,,True,True,,True,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
3,4,easy,Show all products in the 'Home & Kitchen' cate...,SELECT * FROM public.products WHERE products.c...,SELECT * FROM public.products WHERE category =...,24476,,True,True,,False,True,ok,True,"[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738...","[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738..."
4,5,easy,Retrieve all customers who live in the state '...,SELECT * FROM customers WHERE state = 'CA';,SELECT * FROM public.customers WHERE state = '...,24620,,True,True,,False,True,ok,True,"[{""customer_id"": ""6074865c-206a-44a4-9826-84dc...","[{""customer_id"": ""6074865c-206a-44a4-9826-84dc..."
5,6,easy,Get all orders with status 'delivered'.,SELECT * FROM orders WHERE status = 'delivered';,SELECT * FROM public.orders WHERE status = 'de...,24684,,True,True,,False,True,ok,True,"[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23...","[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23..."
6,7,easy,List all order items for the laptop product wh...,SELECT oi.* FROM order_items oi JOIN products ...,SELECT * FROM public.order_items JOIN public.p...,31098,,True,True,,False,False,column mismatch,False,"[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b...","[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b..."
7,8,easy,Show all product reviews with a rating of 5.,SELECT * FROM reviews WHERE rating = 5;,SELECT * FROM public.reviews JOIN public.produ...,32270,,True,True,,False,False,column mismatch,False,"[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874...","[{""review_id"": ""373586a5-73b3-49ea-be5f-799105..."
8,9,easy,Get the distinct product categories available ...,SELECT DISTINCT category FROM products;,SELECT DISTINCT category FROM public.products;,32010,,True,True,,False,True,ok,True,"[{""category"": ""Home & Kitchen""}, {""category"": ...","[{""category"": ""Home & Kitchen""}, {""category"": ..."
9,10,easy,Count the total number of orders.,SELECT COUNT(*) FROM orders;,SELECT count(*) FROM public.orders;,21992,,True,True,,False,True,ok,True,"[{""count"": ""15""}]","[{""count"": ""15""}]"


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.16,1.00,0.76,0.76,25877.24
2,medium,25,0.00,0.76,0.24,0.24,32958.36
1,hard,25,0.00,0.88,0.08,0.08,36628.04


In [19]:
# Run Experiment 3 (Execution-Guided Retry)
df_e3 = run_experiment_3(df_tests)
summary_e3 = save_experiment_results(df_e3, 3)
display(df_e3.head(10))
display(summary_e3)

Semantic enrichment applied ✅


E3: 100%|███████████████████████████████████████| 75/75 [14:24<00:00, 11.52s/it]

Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_3_results.csv
Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_3_summary.csv


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,expected_exec_ok,generated_exec_ok,generated_exec_error,exact_match,result_match_sample,result_diff_hint,generated_ok,expected_result_sample_json,generated_result_sample_json
0,1,easy,List all products.,SELECT * FROM products;,SELECT * FROM public.products;,20910,,True,True,,False,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,SELECT * FROM public.products WHERE category =...,20393,,True,True,,True,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,SELECT * FROM public.products WHERE price > 100;,21590,,True,True,,True,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
3,4,easy,Show all products in the 'Home & Kitchen' cate...,SELECT * FROM public.products WHERE products.c...,SELECT * FROM public.products WHERE category =...,24702,,True,True,,False,True,ok,True,"[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738...","[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738..."
4,5,easy,Retrieve all customers who live in the state '...,SELECT * FROM customers WHERE state = 'CA';,SELECT * FROM public.customers WHERE state = '...,24944,,True,True,,False,True,ok,True,"[{""customer_id"": ""6074865c-206a-44a4-9826-84dc...","[{""customer_id"": ""6074865c-206a-44a4-9826-84dc..."
5,6,easy,Get all orders with status 'delivered'.,SELECT * FROM orders WHERE status = 'delivered';,SELECT * FROM public.orders WHERE status = 'de...,24945,,True,True,,False,True,ok,True,"[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23...","[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23..."
6,7,easy,List all order items for the laptop product wh...,SELECT oi.* FROM order_items oi JOIN products ...,SELECT * FROM public.order_items JOIN public.p...,30163,,True,True,,False,False,column mismatch,False,"[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b...","[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b..."
7,8,easy,Show all product reviews with a rating of 5.,SELECT * FROM reviews WHERE rating = 5;,SELECT * FROM public.reviews JOIN public.produ...,31288,,True,True,,False,False,column mismatch,False,"[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874...","[{""review_id"": ""373586a5-73b3-49ea-be5f-799105..."
8,9,easy,Get the distinct product categories available ...,SELECT DISTINCT category FROM products;,SELECT DISTINCT category FROM public.products;,31044,,True,True,,False,True,ok,True,"[{""category"": ""Home & Kitchen""}, {""category"": ...","[{""category"": ""Home & Kitchen""}, {""category"": ..."
9,10,easy,Count the total number of orders.,SELECT COUNT(*) FROM orders;,SELECT count(*) FROM public.orders;,22203,,True,True,,False,True,ok,True,"[{""count"": ""15""}]","[{""count"": ""15""}]"


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.16,1.00,0.76,0.76,26036.28
2,medium,25,0.00,0.88,0.24,0.24,36975.64
1,hard,25,0.00,0.92,0.12,0.12,39754.28


In [20]:
# Run Experiment 4 (SQL-Specialized Model - sqlcoder)
# Requires: ollama pull sqlcoder  (or set OLLAMA_MODEL_E4 to your Ollama model name)
df_e4 = run_experiment_4(df_tests, retry_enabled=True)
summary_e4 = save_experiment_results(df_e4, 4)
display(df_e4.head(10))
display(summary_e4)

Semantic enrichment applied ✅


E4: 100%|███████████████████████████████████████| 75/75 [06:12<00:00,  4.97s/it]

Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_4_results.csv
Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_4_summary.csv


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,expected_exec_ok,generated_exec_ok,generated_exec_error,exact_match,result_match_sample,result_diff_hint,generated_ok,expected_result_sample_json,generated_result_sample_json
0,1,easy,List all products.,SELECT * FROM products;,,14671,Expecting value: line 2 column 46 (char 47),True,False,Empty SQL,False,False,empty sql,False,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",null
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,,16092,Expecting value: line 2 column 16 (char 17),True,False,Empty SQL,False,False,empty sql,False,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",null
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,,12641,Expecting value: line 2 column 52 (char 53),True,False,Empty SQL,False,False,empty sql,False,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",null
3,4,easy,Show all products in the 'Home & Kitchen' cate...,SELECT * FROM public.products WHERE products.c...,,3969,No JSON found,True,False,Empty SQL,False,False,empty sql,False,"[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738...",null
4,5,easy,Retrieve all customers who live in the state '...,SELECT * FROM customers WHERE state = 'CA';,,4318,Expecting value: line 2 column 17 (char 18),True,False,Empty SQL,False,False,empty sql,False,"[{""customer_id"": ""6074865c-206a-44a4-9826-84dc...",null
5,6,easy,Get all orders with status 'delivered'.,SELECT * FROM orders WHERE status = 'delivered';,,4632,Expecting value: line 2 column 14 (char 15),True,False,Empty SQL,False,False,empty sql,False,"[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23...",null
6,7,easy,List all order items for the laptop product wh...,SELECT oi.* FROM order_items oi JOIN products ...,,14677,'list' object has no attribute 'get',True,False,Empty SQL,False,False,empty sql,False,"[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b...",null
7,8,easy,Show all product reviews with a rating of 5.,SELECT * FROM reviews WHERE rating = 5;,,16510,'list' object has no attribute 'get',True,False,Empty SQL,False,False,empty sql,False,"[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874...",null
8,9,easy,Get the distinct product categories available ...,SELECT DISTINCT category FROM products;,,5432,Expecting value: line 2 column 42 (char 43),True,False,Empty SQL,False,False,empty sql,False,"[{""category"": ""Home & Kitchen""}, {""category"": ...",null
9,10,easy,Count the total number of orders.,SELECT COUNT(*) FROM orders;,,23020,'list' object has no attribute 'get',True,False,Empty SQL,False,False,empty sql,False,"[{""count"": ""15""}]",null


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.0,0.0,0.0,0.0,10494.52
2,medium,25,0.0,0.0,0.0,0.0,12652.48
1,hard,25,0.0,0.0,0.0,0.0,20125.04


In [21]:
# Run Experiment 5 (Example-Augmented KG Few-Shot)
df_e5 = run_experiment_5(df_tests)
summary_e5 = save_experiment_results(df_e5, 5)
display(df_e5.head(10))
display(summary_e5)

Semantic enrichment applied ✅


E5: 100%|███████████████████████████████████████| 75/75 [18:24<00:00, 14.72s/it]

Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_5_results.csv
Saved: /Users/nileshkundu/workspace/db-chat-bot/db-chat-bot-app/test_data_set/nl2sql_experiment_5_summary.csv


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,expected_exec_ok,generated_exec_ok,generated_exec_error,exact_match,result_match_sample,result_diff_hint,generated_ok,expected_result_sample_json,generated_result_sample_json
0,1,easy,List all products.,SELECT * FROM products;,SELECT * FROM public.products;,29511,,True,True,,False,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,SELECT p.* FROM public.products p WHERE p.cate...,30261,,True,True,,False,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,SELECT * FROM public.products WHERE price > 100;,28975,,True,True,,True,True,ok,True,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964..."
3,4,easy,Show all products in the 'Home & Kitchen' cate...,SELECT * FROM public.products WHERE products.c...,SELECT * FROM public.products WHERE category =...,30584,,True,True,,False,True,ok,True,"[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738...","[{""product_id"": ""ae08f3fd-b68f-474f-bb55-c0738..."
4,5,easy,Retrieve all customers who live in the state '...,SELECT * FROM customers WHERE state = 'CA';,SELECT * FROM public.customers WHERE state = '...,31357,,True,True,,False,True,ok,True,"[{""customer_id"": ""6074865c-206a-44a4-9826-84dc...","[{""customer_id"": ""6074865c-206a-44a4-9826-84dc..."
5,6,easy,Get all orders with status 'delivered'.,SELECT * FROM orders WHERE status = 'delivered';,SELECT * FROM public.orders WHERE status = 'de...,31666,,True,True,,False,True,ok,True,"[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23...","[{""order_id"": ""6a7eabe5-d54d-4945-8324-6454b23..."
6,7,easy,List all order items for the laptop product wh...,SELECT oi.* FROM order_items oi JOIN products ...,SELECT * FROM public.order_items oi JOIN publi...,38532,,True,True,,False,False,column mismatch,False,"[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b...","[{""order_item_id"": ""20fc69aa-ef12-4b65-9b96-7b..."
7,8,easy,Show all product reviews with a rating of 5.,SELECT * FROM reviews WHERE rating = 5;,SELECT * FROM public.reviews WHERE rating = 5;,38764,,True,True,,False,True,ok,True,"[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874...","[{""review_id"": ""4e3d2a97-ee4a-4b53-abdd-a57874..."
8,9,easy,Get the distinct product categories available ...,SELECT DISTINCT category FROM products;,SELECT DISTINCT category FROM public.products;,38490,,True,True,,False,True,ok,True,"[{""category"": ""Home & Kitchen""}, {""category"": ...","[{""category"": ""Home & Kitchen""}, {""category"": ..."
9,10,easy,Count the total number of orders.,SELECT COUNT(*) FROM orders;,SELECT COUNT(order_id) FROM public.orders;,28741,,True,True,,False,True,ok,True,"[{""count"": ""15""}]","[{""count"": ""15""}]"


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.12,1.0,0.76,0.76,33739.28
2,medium,25,0.00,1.0,0.36,0.36,42916.96
1,hard,25,0.00,1.0,0.20,0.20,54848.60


## 14. Results Comparison

Compare pass rate, execution success, and result accuracy across experiments.

In [22]:
# Aggregate comparison (run after E1, E2, E3 cells)
rows = []
for name, var in [("E1 Baseline", "df_e1"), ("E2 Semantic+MultiStage", "df_e2"), ("E3 Execution-Guided Retry", "df_e3"), ("E4 SQL-Specialized", "df_e4"), ("E5 Example-Augmented", "df_e5")]:
    df = globals().get(var)
    if df is not None and isinstance(df, pd.DataFrame) and len(df) > 0:
        rows.append({
            "Experiment": name,
            "Pass Rate": df["generated_ok"].mean() * 100,
            "Exec OK Rate": df["generated_exec_ok"].mean() * 100,
            "Result Match Rate": df["result_match_sample"].mean() * 100,
            "Avg Latency (ms)": df["llm_latency_ms"].mean(),
        })
comparison = pd.DataFrame(rows)
if len(comparison) > 0:
    display(comparison)
    comparison.to_csv(os.path.join(RESULTS_DIR, "nl2sql_experiments_comparison.csv"), index=False)
    print("Saved comparison to nl2sql_experiments_comparison.csv")
else:
    print("Run E1, E2, E3 cells first.")

,Experiment,Pass Rate,Exec OK Rate,Result Match Rate,Avg Latency (ms)
0,E1 Baseline,36.000000,98.666667,36.000000,5931.386667
1,E2 Semantic+MultiStage,36.000000,88.000000,36.000000,31821.213333
2,E3 Execution-Guided Retry,37.333333,93.333333,37.333333,34255.400000
3,E4 SQL-Specialized,0.000000,0.000000,0.000000,14424.013333
4,E5 Example-Augmented,44.000000,100.000000,44.000000,43834.946667


Saved comparison to nl2sql_experiments_comparison.csv


## 15. Usage Notes

1. **Prerequisites:** PostgreSQL and Neo4j; Ollama with `gemma3:4b`; for E4: `ollama pull sqlcoder`
2. **Run order:** Top to bottom. E2/E3 enrich KG; E4 uses sqlcoder; E5 seeds 20 examples.
3. **Outputs:** `nl2sql_experiment_{N}_results.csv`, `_summary.csv`, `nl2sql_experiments_comparison.csv`
4. **Customization:** OLLAMA_MODEL, OLLAMA_MODEL_E4, CSV_PATH, retry_enabled (E4)